# Kakamega CCMm analysis

Run the cells **top to bottom** (Shift+Enter, or *Run All*). No command line, no `main()`, so nothing clashes with Jupyter. The only line you may need to change is `INPUT_FILE` in cell 2.

Covers: headline rates, monthly trend, sub-county breakdown, data-quality and zero/missing checks, **population-adjusted incidence**, **age-disaggregated testing & positivity**, and **ward/CHU hotspots with a monthly supervision shortlist**. Produces 13 charts, a `summary.json`, and two supervision CSVs.

**Requirements:** `pip install pandas numpy matplotlib openpyxl`

### 1. Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                       # no display needed; we save to file
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

### 2. Settings, column names & population

Edit `INPUT_FILE` if your file name or location differs. `POP` holds the 2019 census population mapped to each malaria sub-county (Matete folded into Lugari).

In [ ]:
# ====== EDIT THIS IF NEEDED ======
# Your Excel file (exact name, spaces included). Use a full path if it sits
# elsewhere, e.g. r"C:\Users\ADMIN\Downloads\file.xlsx"
INPUT_FILE = "Kakamega CCMm April 2025 - May 2026.xlsx"
SHEET      = "Data"
OUTDIR     = "ccmm_output"
# =================================

COL = {
    "period": "Period",
    "subcounty": "Sub County",
    "ward": "Ward",
    "facility": "Link Facility",
    "chu": "CHU Name",
    "pos": "Total Positive",
    "neg": "Total Negative",
    "not_tested": "Total Not  Tested",
    "suspected": "Total Suspected",
    "invalid": "Total Invalid",
    "tests": "Total Tests",
    "treated": "Total Treated",
    "pos_u5": "CHEW No. positive < 5",
    "pos_o5": "CHEW No. positive >= 5",
    "sus_u5": "CHEW Total suspected < 5",
    "sus_o5": "CHEW Total suspected >= 5",
    "wb1": "CHEW Weight band 5 to <15 kg  (<3 yrs)",
    "wb2": "CHEW Weight band 15 to <25 kg  (3 to <8 yrs)",
    "wb3": "CHEW Weight band 25 to <35 kg (8 to <12 yrs)",
    "wb4": "CHEW Weight band \u2265 35 kg (\u2265 12 yrs)",
}

# Chart palette (kept consistent across every figure)
BLUE, BLUE_L = "#185FA5", "#378ADD"
CORAL, CORAL_D = "#D85A30", "#993C1D"
TEAL, TEAL_D = "#1D9E75", "#0F6E56"
GRAY, GRAY_D = "#9C9A90", "#444441"

# 2019 census population by malaria/health sub-county.
# The census table uses former-district names; they are mapped here to the
# current sub-county names used in the malaria data. Matete (66,172) has no
# malaria sub-county of its own and is folded into Lugari (per the county's own
# table), so the 12 values below sum exactly to the county total of 1,867,579.
POP = {
    "Butere": 154100, "Khwisero": 113476, "Likuyani": 152055,
    "Lugari": 188900,            # Lugari + Matete
    "Matungu": 166940, "Mumias East": 116851, "Mumias West": 115354,
    "Navakholo": 153977,
    "Lurambi": 188212,           # former Kakamega Central
    "Malava": 238330,            # former Kakamega North
    "Shinyalu": 167641,          # former Kakamega East
    "Ikolomani": 111743,         # former Kakamega South
}

### 3. Loading & helper functions

In [ ]:
def pct(n, d):
    return (n / d * 100) if d else 0.0


def load_and_clean(path, sheet):
    """Read the workbook and coerce the count columns to numbers.

    Blank cells become 0 for totals. We keep a separate untouched copy so the
    consistency checks can see the data as entered.
    """
    xls = pd.ExcelFile(path)
    if sheet not in xls.sheet_names:
        raise ValueError(f"Sheet '{sheet}' not found in {path}.\n"
                         f"Available sheets: {xls.sheet_names}")
    df = pd.read_excel(xls, sheet_name=sheet)

    missing = [c for c in COL.values() if c not in df.columns]
    if missing:
        raise KeyError(
            "These expected columns were not found in the sheet:\n  - "
            + "\n  - ".join(missing)
            + "\n\nThe sheet actually contains:\n  - "
            + "\n  - ".join(map(str, df.columns))
            + "\n\nFix: update the COL dictionary to match your column names exactly."
        )
    df = df.dropna(subset=[COL["period"], COL["subcounty"]]).reset_index(drop=True)

    numeric_cols = [COL[k] for k in (
        "pos", "neg", "not_tested", "suspected", "invalid", "tests", "treated",
        "pos_u5", "pos_o5", "sus_u5", "sus_o5", "wb1", "wb2", "wb3", "wb4")]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    # Clean sub-county labels and a chronological ordering of periods.
    df["_sc"] = df[COL["subcounty"]].str.replace(" Sub County", "", regex=False)
    df["_date"] = pd.to_datetime(df[COL["period"]], format="%B %Y", errors="coerce")
    # Drop footer/blank rows that have an unparseable period (e.g. "Grand Total" row).
    # Safe for Kakamega (all periods parse); fixes Kisii which has a footer row.
    df = df[df["_date"].notna()].copy()
    return df


def ordered_periods(df):
    """Return period labels sorted chronologically, plus short axis labels."""
    order = (df[[COL["period"], "_date"]].drop_duplicates()
             .sort_values("_date")[COL["period"]].tolist())
    short = [m.split()[0][:3] + " '" + m.split()[1][2:] for m in order]
    return order, short

### 4. Summary calculations

In [ ]:
def build_summary(df, order):
    """Compute every metric used in the report; return a plain dict + heatmap."""
    S = {}

    # Scope
    S["n_records"] = len(df)
    S["n_months"] = df[COL["period"]].nunique()
    S["n_subcounties"] = df[COL["subcounty"]].nunique()
    S["n_wards"] = df[COL["ward"]].nunique()
    S["n_facilities"] = df[COL["facility"]].nunique()
    S["n_chus"] = df[COL["chu"]].nunique()

    # Headline totals (left key = label used downstream, right = column key in COL)
    total_map = {"suspected": "suspected", "tested": "tests", "not_tested": "not_tested",
                 "pos": "pos", "neg": "neg", "invalid": "invalid", "treated": "treated"}
    tot = {tk: df[COL[ck]].sum() for tk, ck in total_map.items()}
    S["totals"] = tot
    S["testing_rate"] = pct(tot["tested"], tot["suspected"])
    S["positivity"] = pct(tot["pos"], tot["tested"])
    S["treatment_cov"] = pct(tot["treated"], tot["pos"])

    # Age split (overall)
    S["age"] = {
        "sus_u5": df[COL["sus_u5"]].sum(), "sus_o5": df[COL["sus_o5"]].sum(),
        "pos_u5": df[COL["pos_u5"]].sum(), "pos_o5": df[COL["pos_o5"]].sum(),
    }

    # Weight-band (treatment) mix
    S["weight_bands"] = {
        "5 to <15 kg (<3 yrs)": df[COL["wb1"]].sum(),
        "15 to <25 kg (3-<8 yrs)": df[COL["wb2"]].sum(),
        "25 to <35 kg (8-<12 yrs)": df[COL["wb3"]].sum(),
        ">=35 kg (>=12 yrs)": df[COL["wb4"]].sum(),
    }

    # Monthly trend (chronological)
    g = df.groupby(COL["period"])
    monthly = []
    for m in order:
        s = g.get_group(m)
        su, te, po = s[COL["suspected"]].sum(), s[COL["tests"]].sum(), s[COL["pos"]].sum()
        monthly.append(dict(
            month=m, suspected=su, tested=te, positive=po,
            negative=s[COL["neg"]].sum(), treated=s[COL["treated"]].sum(),
            pos_u5=s[COL["pos_u5"]].sum(), pos_o5=s[COL["pos_o5"]].sum(),
            positivity=pct(po, te), testing=pct(te, su),
            active=int((s[COL["suspected"]] > 0).sum()), units=len(s)))
    S["monthly"] = monthly

    # Sub-county breakdown, ranked by positivity
    sub = []
    for sc, s in df.groupby("_sc"):
        su, te, po = s[COL["suspected"]].sum(), s[COL["tests"]].sum(), s[COL["pos"]].sum()
        sub.append(dict(sc=sc, suspected=su, tested=te, positive=po,
                        positivity=pct(po, te), testing=pct(te, su)))
    S["subcounty"] = sorted(sub, key=lambda r: -r["positivity"])

    # Heatmap matrix: sub-county (by overall positivity) x month positivity
    sc_order = [r["sc"] for r in S["subcounty"]]
    mat = np.full((len(sc_order), len(order)), np.nan)
    for i, sc in enumerate(sc_order):
        for j, m in enumerate(order):
            s = df[(df["_sc"] == sc) & (df[COL["period"]] == m)]
            te, po = s[COL["tests"]].sum(), s[COL["pos"]].sum()
            if te:
                mat[i, j] = po / te * 100
    S["heatmap_sc"] = sc_order

    # ---- Consistency / discrepancy checks --------------------------------- #
    calc_tests = df[COL["pos"]] + df[COL["neg"]] + df[COL["invalid"]]
    calc_suspected = df[COL["tests"]] + df[COL["not_tested"]]
    tr_rate = np.where(df[COL["suspected"]] > 0, df[COL["tests"]] / df[COL["suspected"]], np.nan)
    treat_gap = df[COL["treated"]] - df[COL["pos"]]
    count_cols = [COL[k] for k in ("pos", "neg", "not_tested", "suspected", "tests", "treated")]

    S["discrepancies"] = {
        "tests_neq_components": int((calc_tests != df[COL["tests"]]).sum()),
        "suspected_neq_components": int((calc_suspected != df[COL["suspected"]]).sum()),
        "testing_rate_over_100": int(np.nansum(tr_rate > 1.0001)),
        "treated_gt_positive": int((treat_gap > 0).sum()),
        "treated_lt_positive": int((treat_gap < 0).sum()),
        "treated_eq_positive": int((treat_gap == 0).sum()),
        "net_excess_treatment": int(treat_gap.sum()),
        "sum_over_treatment": int(treat_gap.clip(lower=0).sum()),
        "sum_under_treatment": int((-treat_gap).clip(lower=0).sum()),
        "pos_age_mismatch": int(((df[COL["pos_u5"]] + df[COL["pos_o5"]]) != df[COL["pos"]]).sum()),
        "rows_with_fractional_counts": int((df[count_cols] % 1 != 0).any(axis=1).sum()),
        "div0_testing_rate_cells": int((df[COL["suspected"]] == 0).sum()),
        "div0_treatment_cells": int((df[COL["pos"]] == 0).sum()),
    }

    # ---- Zero / missing reporting ----------------------------------------- #
    zero_mask = df[COL["suspected"]] == 0
    by_sc_total = df.groupby("_sc").size()
    by_sc_zero = df[zero_mask].groupby("_sc").size()
    zero_by_sc = [(sc, int(by_sc_zero.get(sc, 0)), int(by_sc_total[sc]),
                   pct(by_sc_zero.get(sc, 0), by_sc_total[sc])) for sc in by_sc_total.index]

    # Sub-county-months with no tests at all (fully inactive periods)
    inactive = []
    for sc in by_sc_total.index:
        for m in order:
            s = df[(df["_sc"] == sc) & (df[COL["period"]] == m)]
            if s[COL["tests"]].sum() == 0:
                inactive.append((sc, m))

    # CHUs that were silent across the entire period
    chu_totals = df.groupby(["_sc", COL["chu"]])[COL["suspected"]].sum()

    S["zero"] = {
        "zero_records": int(zero_mask.sum()),
        "zero_pct": pct(zero_mask.sum(), len(df)),
        "by_subcounty": sorted(zero_by_sc, key=lambda x: -x[3]),
        "fully_inactive_sc_months": inactive,
        "chronic_silent_chus": int((chu_totals == 0).sum()),
        "total_chus": int(len(chu_totals)),
    }
    return S, mat

### 5. Console summary

In [ ]:
def print_summary(S):
    t = S["totals"]
    L = "=" * 64
    print(L); print("SCOPE")
    print(f"  {S['n_records']:,} records | {S['n_months']} months | "
          f"{S['n_subcounties']} sub-counties | {S['n_wards']} wards | "
          f"{S['n_chus']} CHUs | {S['n_facilities']} facilities")
    print(L); print("HEADLINE TOTALS")
    for k, lab in [("suspected", "Suspected"), ("tested", "Tested"),
                   ("not_tested", "Not tested"), ("pos", "Positive"),
                   ("neg", "Negative"), ("invalid", "Invalid"), ("treated", "Treated")]:
        print(f"  {lab:<12}{t[k]:>12,.0f}")
    print(f"  Testing rate        {S['testing_rate']:5.1f}%")
    print(f"  Positivity (TPR)    {S['positivity']:5.1f}%")
    print(f"  Treatment coverage  {S['treatment_cov']:5.1f}%")
    print(L); print("SUB-COUNTY (ranked by positivity)")
    print(f"  {'Sub-county':<14}{'Positive':>10}{'TPR':>8}{'Testing':>9}")
    for r in S["subcounty"]:
        print(f"  {r['sc']:<14}{r['positive']:>10,.0f}{r['positivity']:>7.1f}%{r['testing']:>8.1f}%")
    print(L); print("DISCREPANCIES")
    for k, v in S["discrepancies"].items():
        print(f"  {k:<28}{v:>10,}")
    print(L); print("ZERO / MISSING")
    z = S["zero"]
    print(f"  Zero-activity records: {z['zero_records']:,} ({z['zero_pct']:.1f}%)")
    print(f"  Chronically silent CHUs: {z['chronic_silent_chus']} of {z['total_chus']}")
    if z["fully_inactive_sc_months"]:
        print("  Fully inactive sub-county-months:")
        for sc, m in z["fully_inactive_sc_months"]:
            print(f"    - {sc}: {m}")
    print(L)

### 6. Incidence, age & hotspot analysis

Population-adjusted incidence, age-disaggregated rates, ward/CHU hotspots, case concentration, and the supervision shortlist logic.

In [ ]:
def incidence_rows(S):
    """Community-confirmed malaria incidence & screening reach per 1,000 / yr."""
    ann = 12.0 / S["n_months"]          # annualise from however many months
    rows = []
    for r in S["subcounty"]:
        pop = POP.get(r["sc"])
        if not pop:
            continue
        rows.append(dict(sc=r["sc"], pop=pop, positive=r["positive"],
                         positivity=r["positivity"],
                         incidence=r["positive"] * ann / pop * 1000,
                         screening=r["suspected"] * ann / pop * 1000))
    rows.sort(key=lambda x: -x["incidence"])
    pop_tot = sum(POP.values())
    pos_tot = sum(r["positive"] for r in S["subcounty"])
    sus_tot = sum(r["suspected"] for r in S["subcounty"])
    county = dict(pop=pop_tot, incidence=pos_tot * ann / pop_tot * 1000,
                  screening=sus_tot * ann / pop_tot * 1000)
    return rows, county


def age_disaggregated(df, order):
    extra = ["CHEW No. negative < 5", "CHEW No. negative >= 5",
             "CHEW No. invalid < 5", "CHEW No. invalid >= 5"]
    for c in extra:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    def block(s):
        pu, po = s[COL["pos_u5"]].sum(), s[COL["pos_o5"]].sum()
        nu, no = s["CHEW No. negative < 5"].sum(), s["CHEW No. negative >= 5"].sum()
        iu, io = s["CHEW No. invalid < 5"].sum(), s["CHEW No. invalid >= 5"].sum()
        su, so = s[COL["sus_u5"]].sum(), s[COL["sus_o5"]].sum()
        tu, to = pu + nu + iu, po + no + io          # tested = pos + neg + invalid
        return {"u5": dict(suspected=su, tested=tu, positive=pu,
                           testing=pct(tu, su), positivity=pct(pu, tu)),
                "o5": dict(suspected=so, tested=to, positive=po,
                           testing=pct(to, so), positivity=pct(po, to))}

    overall = block(df)
    g = df.groupby(COL["period"])
    monthly = []
    for m in order:
        b = block(g.get_group(m))
        monthly.append(dict(month=m,
            testing_u5=b["u5"]["testing"], testing_o5=b["o5"]["testing"],
            pos_u5=b["u5"]["positivity"], pos_o5=b["o5"]["positivity"]))
    return overall, monthly


def ward_rows(df):
    rows = []
    for (sc, w), s in df.groupby([COL["subcounty"], COL["ward"]]):
        po, su, te = s[COL["pos"]].sum(), s[COL["suspected"]].sum(), s[COL["tests"]].sum()
        rows.append(dict(subcounty=str(sc).replace(" Sub County", ""), ward=str(w),
                         positive=po, suspected=su, tests=te,
                         positivity=pct(po, te), testing=pct(te, su)))
    rows.sort(key=lambda x: -x["positive"])
    return rows


def chu_rows(df):
    rows = []
    for (sc, chu), s in df.groupby([COL["subcounty"], COL["chu"]]):
        po, su = s[COL["pos"]].sum(), s[COL["suspected"]].sum()
        te, tr = s[COL["tests"]].sum(), s[COL["treated"]].sum()
        rows.append(dict(
            subcounty=str(sc).replace(" Sub County", ""), chu=str(chu),
            positive=po, suspected=su, tests=te, treated=tr,
            testing=pct(te, su), untreated=max(po - tr, 0),
            impossible=int((s[COL["tests"]] > s[COL["suspected"]]).sum()),
            missing_months=int((s[COL["suspected"]] == 0).sum())))
    rows.sort(key=lambda x: -x["positive"])
    return rows


def concentration(chu):
    """Lorenz-style cumulative share of positives across CHUs (ranked desc)."""
    cc = sorted(chu, key=lambda x: -x["positive"])
    tot = sum(c["positive"] for c in cc) or 1
    n = len(cc)
    cum_units, cum_pos, run = [], [], 0.0
    for i, c in enumerate(cc, 1):
        run += c["positive"]
        cum_units.append(i / n * 100)
        cum_pos.append(run / tot * 100)

    def share_at(frac):
        k = max(1, int(round(n * frac)))
        return sum(c["positive"] for c in cc[:k]) / tot * 100

    return dict(n=n, cum_units=cum_units, cum_pos=cum_pos,
                top10=share_at(0.10), top20=share_at(0.20),
                top30=share_at(0.30), top50=share_at(0.50))


def supervision_lists(chu, n_burden=10, n_ops=12):
    """Two actionable lists for picking where to supervise each month:
       (1) the highest-burden CHUs, (2) CHUs flagged for operational gaps."""
    burden = sorted(chu, key=lambda x: -x["positive"])[:n_burden]
    ops = []
    for c in chu:
        reasons = []
        if c["suspected"] >= 100 and c["testing"] < 70:
            reasons.append(f"low testing ({c['testing']:.0f}%)")
        if c["untreated"] >= 20:
            reasons.append(f"{c['untreated']:.0f} positives untreated")
        if c["missing_months"] >= 3:
            reasons.append(f"{c['missing_months']} months not reported")
        if c["impossible"] >= 3:
            reasons.append(f"{c['impossible']} impossible records")
        if reasons:
            d = dict(c); d["reasons"] = "; ".join(reasons)
            d["score"] = len(reasons) * 100000 + c["suspected"]
            ops.append(d)
    ops.sort(key=lambda x: -x["score"])
    return burden, ops[:n_ops]


def print_extended(df, S, order):
    """Console output for the incidence, age, and hotspot analyses."""
    inc, county = incidence_rows(S)
    overall, _ = age_disaggregated(df, order)
    cr = chu_rows(df); conc = concentration(cr); wr = ward_rows(df)
    _, ops = supervision_lists(cr)
    L = "=" * 64
    print(L); print("POPULATION-ADJUSTED INCIDENCE (community-confirmed, per 1,000/yr)")
    print(f"  {'Sub-county':<13}{'Pop':>9}{'Inc/1k/yr':>11}{'Screen/1k/yr':>13}")
    for r in inc:
        print(f"  {r['sc']:<13}{r['pop']:>9,}{r['incidence']:>11.0f}{r['screening']:>13.0f}")
    print(f"  COUNTY: incidence {county['incidence']:.0f}/1,000/yr | screening {county['screening']:.0f}/1,000/yr")
    print(L); print("AGE-DISAGGREGATED RATES (whole period)")
    for k, lab in [("u5", "Under-5"), ("o5", "5 & over")]:
        b = overall[k]
        print(f"  {lab:<9}: testing {b['testing']:.0f}%   positivity {b['positivity']:.0f}%   (tested {b['tested']:,.0f})")
    print(L); print("HOTSPOTS & SUPERVISION")
    print(f"  Concentration: top 20% of {conc['n']} CHUs -> {conc['top20']:.0f}% of positives; top 50% -> {conc['top50']:.0f}%")
    print("  Highest-burden wards:")
    for r in wr[:6]:
        print(f"    {r['ward'][:24]:<25}{r['subcounty']:<12}{r['positive']:>6,.0f} positives ({r['positivity']:.0f}% TPR)")
    print("  Supervision shortlist (operational gaps):")
    for c in ops[:8]:
        print(f"    {c['chu'][:32]:<33}{c['subcounty']:<12}{c['reasons']}")
    print(L)

### 7. Charts

In [ ]:
def _style():
    plt.rcParams.update({
        "font.family": "DejaVu Sans", "font.size": 10, "figure.dpi": 150,
        "axes.edgecolor": "#666", "axes.linewidth": 0.8,
        "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlecolor": "#222",
        "text.color": "#333", "axes.labelcolor": "#333",
        "xtick.color": "#444", "ytick.color": "#444"})


def _clean(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", color="#000", alpha=0.07, linewidth=0.8)
    ax.set_axisbelow(True)


def _save(fig, outdir, name):
    fig.tight_layout()
    fig.savefig(os.path.join(outdir, name), bbox_inches="tight", facecolor="white")
    plt.close(fig)


def make_charts(S, mat, order, short, outdir):
    _style()
    os.makedirs(outdir, exist_ok=True)
    pctfmt = FuncFormatter(lambda v, _: f"{v:.0f}%")
    intfmt = FuncFormatter(lambda v, _: f"{int(v):,}")

    # 1. Cascade
    fig, ax = plt.subplots(figsize=(8, 3.6))
    t = S["totals"]
    vals = [t["suspected"], t["tested"], t["pos"], t["treated"]]
    steps = ["Suspected", "Tested", "Positive", "Treated"]
    cols = [BLUE, BLUE, CORAL_D, TEAL_D]
    y = np.arange(len(steps))[::-1]
    ax.barh(y, vals, color=cols, height=0.62, zorder=3)
    for yi, v in zip(y, vals):
        ax.text(v + vals[0] * 0.015, yi, f"{int(round(v)):,}  ({v/vals[0]*100:.0f}%)",
                va="center", fontsize=10, fontweight="bold", color="#333")
    ax.set_yticks(y); ax.set_yticklabels(steps); ax.set_xlim(0, vals[0] * 1.22)
    for sp in ("top", "right", "bottom"):
        ax.spines[sp].set_visible(False)
    ax.set_xticks([])
    ax.set_title("Malaria testing-and-treatment cascade", pad=12)
    _save(fig, outdir, "01_cascade.png")

    # 2. Monthly trend
    fig, ax = plt.subplots(figsize=(8, 4.2))
    x = np.arange(len(order))
    tpr = [m["positivity"] for m in S["monthly"]]
    tr = [m["testing"] for m in S["monthly"]]
    ax.plot(x, tpr, "-o", color=CORAL, lw=2.4, ms=5, label="Test positivity rate", zorder=4)
    ax.plot(x, tr, "--s", color=BLUE_L, lw=2.4, ms=5, label="Testing rate", zorder=4)
    for xi, v in zip(x, tpr):
        ax.annotate(f"{v:.0f}", (xi, v), textcoords="offset points", xytext=(0, -14),
                    ha="center", fontsize=8, color=CORAL_D)
    ax.set_ylim(0, 100); ax.set_xticks(x)
    ax.set_xticklabels(short, rotation=45, ha="right", fontsize=9)
    ax.yaxis.set_major_formatter(pctfmt); _clean(ax)
    ax.legend(frameon=False, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.32), fontsize=10)
    ax.set_title("Monthly test positivity vs testing rate", pad=12)
    _save(fig, outdir, "02_monthly_trend.png")

    # 3. Sub-county positivity
    fig, ax = plt.subplots(figsize=(8, 4.6))
    sc = S["subcounty"]; names = [r["sc"] for r in sc]; tprs = [r["positivity"] for r in sc]
    y = np.arange(len(names))[::-1]
    ax.barh(y, tprs, color=CORAL, height=0.7, zorder=3)
    ax.axvline(S["positivity"], color=GRAY_D, ls=":", lw=1.4, zorder=2)
    ax.text(S["positivity"] + 0.5, len(names) - 0.3, f"County avg {S['positivity']:.0f}%",
            fontsize=8.5, color=GRAY_D, va="top")
    for yi, v in zip(y, tprs):
        ax.text(v + 0.8, yi, f"{v:.0f}%", va="center", fontsize=9, fontweight="bold", color="#333")
    ax.set_yticks(y); ax.set_yticklabels(names); ax.set_xlim(0, max(tprs) * 1.15)
    ax.xaxis.set_major_formatter(pctfmt); _clean(ax)
    ax.set_title("Test positivity rate by sub-county", pad=12)
    _save(fig, outdir, "03_subcounty_positivity.png")

    # 4. Heatmap
    fig, ax = plt.subplots(figsize=(9, 5.2))
    cmap = plt.cm.YlOrRd.copy(); cmap.set_bad("#EEEEEE")
    im = ax.imshow(np.ma.masked_invalid(mat), cmap=cmap, aspect="auto", vmin=10, vmax=85)
    ax.set_xticks(np.arange(len(order))); ax.set_xticklabels(short, rotation=45, ha="right", fontsize=8.5)
    ax.set_yticks(np.arange(len(S["heatmap_sc"]))); ax.set_yticklabels(S["heatmap_sc"], fontsize=9)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            if not np.isnan(mat[i, j]):
                ax.text(j, i, f"{mat[i, j]:.0f}", ha="center", va="center", fontsize=7,
                        color="white" if mat[i, j] > 55 else "#3a3a3a")
    cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cb.set_label("Positivity %", fontsize=9); cb.ax.tick_params(labelsize=8)
    ax.set_xticks(np.arange(-.5, len(order), 1), minor=True)
    ax.set_yticks(np.arange(-.5, mat.shape[0], 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.5); ax.tick_params(which="minor", length=0)
    ax.set_title("Test positivity (%) by sub-county and month", pad=12)
    _save(fig, outdir, "04_heatmap.png")

    # 5. Counts vs rate (twin axes)
    fig, ax = plt.subplots(figsize=(9, 4.6))
    scc = sorted(S["subcounty"], key=lambda r: -r["positive"])
    names = [r["sc"] for r in scc]; pos = [r["positive"] for r in scc]; rt = [r["positivity"] for r in scc]
    x = np.arange(len(names))
    ax.bar(x, pos, color=BLUE_L, width=0.62, zorder=3, label="Positive cases (count)")
    ax.set_ylabel("Positive cases", color=BLUE); ax.tick_params(axis="y", labelcolor=BLUE)
    ax.yaxis.set_major_formatter(intfmt)
    ax2 = ax.twinx()
    ax2.plot(x, rt, "-D", color=CORAL, lw=2, ms=6, zorder=5, label="Positivity rate (%)")
    ax2.set_ylabel("Positivity rate", color=CORAL_D); ax2.tick_params(axis="y", labelcolor=CORAL_D)
    ax2.set_ylim(0, max(rt) * 1.2); ax2.yaxis.set_major_formatter(pctfmt)
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=40, ha="right", fontsize=8.5)
    ax.spines["top"].set_visible(False); ax2.spines["top"].set_visible(False)
    ax.set_axisbelow(True); ax.grid(axis="y", color="#000", alpha=0.06)
    l1, la1 = ax.get_legend_handles_labels(); l2, la2 = ax2.get_legend_handles_labels()
    ax.legend(l1 + l2, la1 + la2, frameon=False, loc="upper right", fontsize=9)
    ax.set_title("Burden vs intensity: positive counts and positivity by sub-county", pad=12, fontsize=11.5)
    _save(fig, outdir, "05_counts_vs_rate.png")

    # 6. Age split over time
    fig, ax = plt.subplots(figsize=(8, 4.2))
    u5 = [m["pos_u5"] for m in S["monthly"]]; o5 = [m["pos_o5"] for m in S["monthly"]]
    x = np.arange(len(order))
    ax.bar(x, o5, color=TEAL, width=0.66, zorder=3, label="Positive, >=5 yrs")
    ax.bar(x, u5, bottom=o5, color=CORAL, width=0.66, zorder=3, label="Positive, <5 yrs")
    ax.set_xticks(x); ax.set_xticklabels(short, rotation=45, ha="right", fontsize=9)
    ax.yaxis.set_major_formatter(intfmt); _clean(ax)
    ax.legend(frameon=False, loc="upper left", fontsize=9)
    ax.set_title("Monthly positive cases by age group", pad=12)
    _save(fig, outdir, "06_age_split.png")

    # 7. Treatment mix by weight band
    fig, ax = plt.subplots(figsize=(8, 4.0))
    wb = S["weight_bands"]; labels = list(wb.keys()); vals = list(wb.values())
    total = sum(vals) or 1
    x = np.arange(len(labels))
    ax.bar(x, vals, color=["#85B7EB", "#378ADD", "#185FA5", "#0C447C"], width=0.62, zorder=3)
    for xi, v in zip(x, vals):
        ax.text(xi, v + total * 0.01, f"{int(v):,}\n({v/total*100:.0f}%)",
                ha="center", fontsize=9, fontweight="bold", color="#333")
    ax.set_xticks(x); ax.set_xticklabels([l.replace(" ", "\n", 1) for l in labels], fontsize=8.5)
    ax.set_ylim(0, max(vals) * 1.18); ax.yaxis.set_major_formatter(intfmt); _clean(ax)
    ax.set_title("Treatments dispensed by weight band (Coartem dosing)", pad=12)
    _save(fig, outdir, "07_treatment_mix.png")

    # 8. Reporting completeness
    fig, ax = plt.subplots(figsize=(8, 4.0))
    act = [pct(m["active"], m["units"]) for m in S["monthly"]]
    x = np.arange(len(order))
    ax.plot(x, act, "-o", color=TEAL_D, lw=2.4, ms=5, zorder=4)
    ax.fill_between(x, act, color=TEAL, alpha=0.12, zorder=2)
    for xi, v in zip(x, act):
        ax.annotate(f"{v:.0f}", (xi, v), textcoords="offset points", xytext=(0, 8),
                    ha="center", fontsize=8, color=TEAL_D)
    ax.set_ylim(min(act) - 8, 102); ax.set_xticks(x)
    ax.set_xticklabels(short, rotation=45, ha="right", fontsize=9)
    ax.yaxis.set_major_formatter(pctfmt); _clean(ax)
    ax.set_title("Reporting completeness: share of units reporting activity", pad=12)
    _save(fig, outdir, "08_reporting.png")

    # 9. Monthly volume (stacked composition of suspected)
    fig, ax = plt.subplots(figsize=(8, 4.4))
    posv = [m["positive"] for m in S["monthly"]]
    negv = [m["negative"] for m in S["monthly"]]
    ntv = [m["suspected"] - m["tested"] for m in S["monthly"]]
    x = np.arange(len(order))
    ax.bar(x, posv, color=CORAL, width=0.66, zorder=3, label="Positive")
    ax.bar(x, negv, bottom=posv, color=BLUE_L, width=0.66, zorder=3, label="Negative")
    ax.bar(x, ntv, bottom=np.array(posv) + np.array(negv), color=GRAY, width=0.66, zorder=3, label="Not tested")
    ax.set_xticks(x); ax.set_xticklabels(short, rotation=45, ha="right", fontsize=9)
    ax.yaxis.set_major_formatter(intfmt); _clean(ax)
    ax.legend(frameon=False, loc="upper left", ncol=3, fontsize=9)
    ax.set_title("Monthly suspected caseload by outcome", pad=12)
    _save(fig, outdir, "09_monthly_volume.png")

    print(f"Saved 9 charts to {outdir}/")


def make_extra_charts(df, S, order, outdir):
    """Charts 10-13: incidence, age rates, case concentration, top wards."""
    _style(); os.makedirs(outdir, exist_ok=True)
    pctfmt = FuncFormatter(lambda v, _: f"{v:.0f}%")
    intfmt = FuncFormatter(lambda v, _: f"{int(v):,}")

    # 10. Incidence + screening reach by sub-county (grouped horizontal bars)
    inc, county = incidence_rows(S)
    names = [r["sc"] for r in inc]
    screen = [r["screening"] for r in inc]; incd = [r["incidence"] for r in inc]
    y = np.arange(len(names))[::-1]; h = 0.38
    fig, ax = plt.subplots(figsize=(8.8, 5.0))
    ax.barh(y + h/2, screen, height=h, color=BLUE_L, zorder=3, label="Screened per 1,000 / yr")
    ax.barh(y - h/2, incd, height=h, color=CORAL, zorder=3, label="Confirmed cases per 1,000 / yr")
    ax.axvline(county["incidence"], color=GRAY_D, ls=":", lw=1.4, zorder=2)
    ax.text(county["incidence"] + 3, len(names) - 0.6, f"county avg incidence {county['incidence']:.0f}",
            fontsize=8.5, color=GRAY_D, va="top")
    ax.set_yticks(y); ax.set_yticklabels(names, fontsize=11)
    ax.set_xlabel("per 1,000 population per year")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.grid(axis="x", color="#000", alpha=0.07); ax.set_axisbelow(True)
    ax.legend(frameon=False, loc="lower right", fontsize=10)
    ax.set_title("Community malaria incidence & screening reach by sub-county", pad=12, fontsize=12)
    _save(fig, outdir, "10_incidence.png")

    # 11. Testing rate and positivity by age group
    overall, _ = age_disaggregated(df, order)
    u5 = [overall["u5"]["testing"], overall["u5"]["positivity"]]
    o5 = [overall["o5"]["testing"], overall["o5"]["positivity"]]
    x = np.arange(2); w = 0.36
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    b1 = ax.bar(x - w/2, u5, w, color=CORAL, zorder=3, label="Under-5")
    b2 = ax.bar(x + w/2, o5, w, color=TEAL, zorder=3, label="5 & over")
    for bars in (b1, b2):
        for b in bars:
            ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1.2, f"{b.get_height():.0f}%",
                    ha="center", fontsize=11, fontweight="bold", color="#222")
    ax.set_xticks(x); ax.set_xticklabels(["Testing rate", "Positivity"], fontsize=12)
    ax.set_ylim(0, 100); ax.yaxis.set_major_formatter(pctfmt); _clean(ax)
    ax.legend(frameon=False, fontsize=10)
    ax.set_title("Testing rate and positivity by age group", pad=12)
    _save(fig, outdir, "11_age_rates.png")

    # 12. Case concentration across community units (Lorenz-style)
    cr = chu_rows(df); conc = concentration(cr)
    fig, ax = plt.subplots(figsize=(7.6, 4.6))
    ax.plot([0] + conc["cum_units"], [0] + conc["cum_pos"], color=BLUE, lw=2.6, zorder=4)
    ax.plot([0, 100], [0, 100], color=GRAY, ls="--", lw=1.2, zorder=2)
    for frac, val, lab in [(20, conc["top20"], "top 20%"), (50, conc["top50"], "top 50%")]:
        ax.plot([frac], [val], "o", color=CORAL, ms=7, zorder=5)
        ax.annotate(f"{lab}: {val:.0f}%", (frac, val), textcoords="offset points",
                    xytext=(8, -4), fontsize=10, color=CORAL_D, fontweight="bold")
    ax.set_xlim(0, 100); ax.set_ylim(0, 100)
    ax.set_xlabel("Cumulative share of community units (ranked by burden)")
    ax.set_ylabel("Cumulative share of positives")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.grid(color="#000", alpha=0.07); ax.set_axisbelow(True)
    ax.set_title("Case concentration across community units", pad=12)
    _save(fig, outdir, "12_concentration.png")

    # 13. Highest-burden wards
    top = ward_rows(df)[:12]
    names = [r["ward"].replace(" Ward", "") for r in top][::-1]
    pos = [r["positive"] for r in top][::-1]
    y = np.arange(len(names))
    fig, ax = plt.subplots(figsize=(8.6, 5.0))
    ax.barh(y, pos, color=CORAL, height=0.72, zorder=3)
    for yi, v in zip(y, pos):
        ax.text(v + max(pos)*0.01, yi, f"{v:,.0f}", va="center", fontsize=9, color="#222")
    ax.set_yticks(y); ax.set_yticklabels(names, fontsize=9.5)
    ax.set_xlabel("Confirmed positive cases (whole period)")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.grid(axis="x", color="#000", alpha=0.07); ax.set_axisbelow(True)
    ax.set_title("Highest-burden wards", pad=12)
    _save(fig, outdir, "13_top_wards.png")

    print(f"Saved 4 incidence/supervision charts to {outdir}/")

### 8. Run everything

Loads the file, prints both summaries, writes `summary.json`, and saves all 13 charts.

In [ ]:
import os, json

os.makedirs(OUTDIR, exist_ok=True)

# 1) load the spreadsheet
df = load_and_clean(INPUT_FILE, SHEET)
order, short = ordered_periods(df)

# 2) core summary + the new incidence / age / hotspot analyses
S, mat = build_summary(df, order)
print_summary(S)
print_extended(df, S, order)

# 3) attach the extra analyses to S and save everything as JSON
inc, county = incidence_rows(S)
overall, age_monthly = age_disaggregated(df, order)
cr = chu_rows(df); conc = concentration(cr); wr = ward_rows(df)
burden, ops = supervision_lists(cr)
S["incidence"] = {"subcounty": inc, "county": county}
S["age_rates"] = {"overall": overall, "monthly": age_monthly}
S["hotspots"] = {"concentration": {k: conc[k] for k in ("n","top10","top20","top30","top50")},
                 "top_wards": wr[:15], "supervision_burden": burden, "supervision_ops": ops}
with open(os.path.join(OUTDIR, "summary.json"), "w") as f:
    json.dump(S, f, indent=2, default=lambda x: round(float(x), 4)
              if isinstance(x, (np.floating, float)) else int(x))

# 4) all 13 charts
make_charts(S, mat, order, short, os.path.join(OUTDIR, "charts"))
make_extra_charts(df, S, order, os.path.join(OUTDIR, "charts"))

### 9. Monthly supervision lists

The two tables to drive supervision visits — highest-burden units and units flagged for operational gaps. Re-run on each new monthly export. Also saved as CSVs.


In [ ]:
import os, pandas as pd
try:
    from IPython.display import display
except Exception:
    display = print

# Re-run this cell each month on the latest export to pick where to supervise.
cr = chu_rows(df)
burden, ops = supervision_lists(cr, n_burden=15, n_ops=9999)

bdf = pd.DataFrame(burden)
bdf["positive"] = bdf["positive"].astype(int); bdf["testing"] = bdf["testing"].round(0).astype(int)
burden_df = bdf[["chu","subcounty","positive","testing","missing_months"]]
burden_df.columns = ["CHU","Sub-county","Positives","Testing %","Missing months"]

odf = pd.DataFrame(ops)
odf["positive"] = odf["positive"].astype(int)
ops_df = odf[["chu","subcounty","positive","reasons"]]
ops_df.columns = ["CHU","Sub-county","Positives","Why flagged"]

print("Highest-burden CHUs (support the high-volume sites):")
display(burden_df)
print("\nSupervision shortlist - operational gaps to visit:")
display(ops_df)

burden_df.to_excel(os.path.join(OUTDIR, "supervision_burden.xlsx"), index=False)
ops_df.to_excel(os.path.join(OUTDIR, "CHU_Data_Quality_Flags.xlsx"), index=False)
print("\nSaved Excel files to", OUTDIR)


### 9b. Chronically silent CHUs

Units that submitted **zero suspected cases for the entire period** — they never appear on the burden or operational-gaps lists because they have no activity to flag. This list surfaces them by name so they can be visited and either onboarded or removed from the register.


In [ ]:
# ── CHRONICALLY SILENT CHUs ─────────────────────────────────────────────────
# Units that submitted ZERO suspected cases across every single month.
# These never appear on the burden or operational-gaps shortlists because
# they have no activity data to flag — so they need a dedicated named list.

cr = chu_rows(df)
n_months = len(order)

silent = [c for c in cr if c['suspected'] == 0]
silent_df = pd.DataFrame([{
    'CHU': c['chu'],
    'Sub-county': c['subcounty'],
    'Months in data': c['missing_months'],
    'Total months': n_months,
    'Positives': int(c['positive']),
    'Action needed': 'Field visit: confirm active/inactive; if active onboard for reporting; if inactive remove from register'
} for c in silent])

print(f"Chronically silent CHUs (zero activity all {n_months} months): {len(silent)}")
display(silent_df)

silent_df.to_excel(os.path.join(OUTDIR, "supervision_silent_chus.xlsx"), index=False)
print("\nSaved to", os.path.join(OUTDIR, "supervision_silent_chus.xlsx"))


### 10. (Optional) show all charts inline

In [ ]:
# Show all charts inline (needs Jupyter)
import glob, os
try:
    from IPython.display import Image, display
    for png in sorted(glob.glob(os.path.join(OUTDIR, "charts", "*.png"))):
        display(Image(filename=png))
except Exception:
    print("Open in Jupyter to see inline charts. They are saved in", os.path.join(OUTDIR, "charts"))